In [ ]:
%run init_env.py

Environment initializing completed successfully.


In [8]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [23]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

llm = create_azure_llm()

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=llm,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [28]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Regards, John."
    },
    config=config,
    version="v2"
)

In [25]:
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'Thank '
                                                                          'you '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'No '
                                                                          'problem, '
                                                                          'we '
                  

In [30]:
response["__interrupt__"]

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John,\n\nThanks for the update. No worries about being late. Please let me know your availability so we can reschedule the meeting at a convenient time.\n\nBest regards,\nSeán'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John,\\n\\nThanks for the update. No worries about being late. Please let me know your availability so we can reschedule the meeting at a convenient time.\\n\\nBest regards,\\nSeán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='0c5318277c69ca0e6365c38e7fe55d55')]

In [29]:
print(response["__interrupt__"][0].value['action_requests'][0]['args']['body'])

Hi John,

Thanks for the update. No worries about being late. Please let me know your availability so we can reschedule the meeting at a convenient time.

Best regards,
Seán


## Approve

In [27]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config, # Same thread ID to resume the paused conversation
    version="v2"
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Regards, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='8b4e2dc5-40d4-454b-8048-d874c026b869'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 86, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DSJk5qv6Hujq9ohqknbKWudBW0Rah', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbr

## Reject

In [26]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Request rejected!."
                }
            ],
        }
    ), 
    config=config, # Same thread ID to resume the paused conversation
    version="v2"

    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'Thank '
                                                                          'you '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'No '
                                                                          'problem, '
                                                                          'we '
                  

## Edit

In [31]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call. # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool. Updated body will generally come from a UI where 
                        # user has edited the email body
                        "args": {"body": "updated email body!"},
                    }
                }
            ]
        }
    ), 
    config=config, # Same thread ID to resume the paused conversation
    version="v2"
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Regards, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='8b4e2dc5-40d4-454b-8048-d874c026b869'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 86, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DSJk5qv6Hujq9ohqknbKWudBW0Rah', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbr